# Building a Demand Series with SQL
### Kestrel Garden & Outdoor: from sale lines to a weekly series

---

Forecasts do not start with a model. They start with a table that has one row per week, and the shop's till does not produce one. It produces one row per sale line: 79,848 of them. Somebody has to turn those lines into weeks without losing any, double counting any, or quietly dropping a week.

That somebody is you, and the tool is SQL. You will:

1. Read four queries that turn sale lines into answers, and say what each one does before you run it.
2. Ask Copilot to write three queries of your own for business questions, and check them.
3. Meet the week that disappears, and see why one join keeps it.

**The data is synthetic.** `sales_lines.csv` and `weekly_calendar.csv` were generated for this module. Kestrel is fictional.

### The AI rule

Copilot is on. It writes SQL quickly and confidently, and it will sometimes write the wrong query. Every query it gives you gets checked against pandas before you believe it.

### How each part works

Every part follows the same five steps: **Load and look**, **Run the query**, **Read the output in plain English**, **Check against pandas**, **Say it to a manager**.

## Setup: the database

SQLite is a small database that lives inside Python. The cell below loads both files into it as two tables, `sales` and `calendar`, so you can query them with SQL.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

sales = pd.read_csv("../data/sales_lines.csv")
calendar = pd.read_csv("../data/weekly_calendar.csv")

con = sqlite3.connect(":memory:")          # a database that lives in memory and disappears when the notebook closes
sales.to_sql("sales", con, index=False)
calendar.to_sql("calendar", con, index=False)

def query(sql, params=()):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(sql, con, params=params)

print("Synthetic data: Kestrel Garden & Outdoor is fictional.")
print(f"sales: {len(sales):,} rows | calendar: {len(calendar)} rows")

In [ ]:
# What the columns look like before any query
print(query("SELECT * FROM sales LIMIT 5").to_string(index=False))
print()
print(query("SELECT * FROM calendar LIMIT 5").to_string(index=False))

---
## Part 1: Demo: GROUP BY, dates and the calendar join

**The business question:** how many units, and how much money, does each category bring in, and how do we get to a weekly series for barbecues?

### Step 1: Load and look

Each row of `sales` is one line on one receipt: a date, a category, a quantity and a unit price. Before grouping, find out what you are grouping.

In [ ]:
print(query("SELECT MIN(SaleDate) AS first_sale, MAX(SaleDate) AS last_sale, COUNT(*) AS lines, SUM(Quantity) AS units FROM sales").to_string(index=False))
print(query("SELECT DISTINCT Category FROM sales ORDER BY Category").to_string(index=False))

### Step 2: Run the query

Four queries. **Read each one aloud, line by line, and say what it does before you run it.** The idea to hold on to: `GROUP BY` squashes many rows into one row per group, and the functions like `SUM` say how.

In [ ]:
# Query 1: units and revenue by category
Q1 = """
SELECT Category,
       SUM(Quantity)             AS units,
       ROUND(SUM(Quantity * UnitPrice), 0) AS revenue
FROM sales
GROUP BY Category
ORDER BY revenue DESC
"""
by_category = query(Q1)
print(by_category.to_string(index=False))

In [ ]:
# Query 2: units by calendar year. Careful with the last row.
Q2 = """
SELECT strftime('%Y', SaleDate) AS year,
       SUM(Quantity)            AS units
FROM sales
GROUP BY year
ORDER BY year
"""
by_year = query(Q2)
print(by_year.to_string(index=False))

In [ ]:
# Query 3: units by week. date(SaleDate, 'weekday 0', '-6 days') moves any date to the Monday that starts its week.
Q3 = """
SELECT date(SaleDate, 'weekday 0', '-6 days') AS week_start,
       SUM(Quantity)                          AS units
FROM sales
WHERE Category = 'Barbecue and Dining'
GROUP BY week_start
ORDER BY week_start
"""
barbecue_by_week = query(Q3)
print(barbecue_by_week.head(5).to_string(index=False))
print("Weeks returned:", len(barbecue_by_week), "| total units:", barbecue_by_week["units"].sum())

In [ ]:
# Query 4: units by weekday. strftime('%w') gives 0 for Sunday to 6 for Saturday.
Q4 = """
SELECT CASE strftime('%w', SaleDate)
           WHEN '0' THEN 'Sunday' WHEN '1' THEN 'Monday' WHEN '2' THEN 'Tuesday'
           WHEN '3' THEN 'Wednesday' WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
           ELSE 'Saturday' END AS weekday,
       SUM(Quantity) AS units,
       ROUND(100.0 * SUM(Quantity) / (SELECT SUM(Quantity) FROM sales), 1) AS share_percent
FROM sales
GROUP BY strftime('%w', SaleDate)
ORDER BY units DESC
"""
by_weekday = query(Q4)
print(by_weekday.to_string(index=False))

### Step 3: Read the output in plain English

The cell below writes the answers in words from the tables above.

In [ ]:
top = by_category.iloc[0]
print(f"{top['Category']} earns the most: £{top['revenue']:,.0f} from {top['units']:,} units.")
print(f"Most units are not in the category with the most revenue: the units leader is {by_category.sort_values('units').iloc[-1]['Category']}.")
last = by_year.iloc[-1]
print(f"The last year in the file is {last['year']} with only {last['units']:,} units. That is the last week of December spilling into January, not a collapse in sales.")
best_day = by_weekday.iloc[0]
print(f"{best_day['weekday']} is the busiest day, with {best_day['share_percent']}% of all units.")

**Why group by week start and not by year?** A week that runs from Monday 29 December to Sunday 4 January belongs to one week. Grouping by calendar year splits it in two. For a weekly forecast, the unit of analysis is the week.

### Step 4: Check against pandas

Run the same questions in pandas and compare. SQL and pandas should agree to the unit. When they do not, one of them is answering a different question from the one you thought you asked.

In [ ]:
pd_category = sales.assign(revenue=sales["Quantity"] * sales["UnitPrice"]).groupby("Category").agg(units=("Quantity", "sum"), revenue=("revenue", "sum"))
checks = {
    "units by category match": bool((pd_category["units"].sort_index().to_numpy() == by_category.sort_values("Category")["units"].to_numpy()).all()),
    "revenue by category match": bool(np.allclose(pd_category["revenue"].sort_index().round(0).to_numpy(), by_category.sort_values("Category")["revenue"].to_numpy())),
    "year totals match": int(by_year["units"].sum()) == int(sales["Quantity"].sum()),
    "barbecue weekly units add up to the barbecue total": int(barbecue_by_week["units"].sum()) == int(sales.loc[sales["Category"] == "Barbecue and Dining", "Quantity"].sum()),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

### Step 5: Say it to a manager

Two sentences for the buying director. Draft below; rewrite in your own words underneath.

In [ ]:
print(f"Barbecue and Dining brings in £{by_category.iloc[0]['revenue']:,.0f}, more than any other category, and {best_day['share_percent']:.0f}% of all units are sold on a {best_day['weekday']}.")
print("The weekly series for barbecues adds up to the till total, so we can forecast from it.")

*Your two sentences here*

### Commentary: Part 1

1. In Query 3, what does `GROUP BY week_start` do to 53,235 units' worth of rows?
2. Why does the 2026 row in Query 2 not mean sales fell?
3. Why check SQL against pandas when SQL already gave an answer?

*Your commentary here*

### Your turn: change the query

In Query 3, change `'Barbecue and Dining'` to `'Tools'`, run it, and read the total. Does it match the Tools units in Query 1? Put it back afterwards.

---
## Part 2: Demo: the week that disappears

**The business question:** the forecast needs every week, including a week in which nothing sold. What does GROUP BY do with a week that has no sale lines?

### Step 1: Load and look

The real data has sales in every week, so to see the problem we need a gap. The cell below makes a copy of the sales table called `sales_gap`, with every line from the week starting 23 December 2024 removed. Think of it as a week when the tills were down.

In [ ]:
con.execute("DROP TABLE IF EXISTS sales_gap")
con.execute("""
CREATE TABLE sales_gap AS
SELECT * FROM sales
WHERE date(SaleDate, 'weekday 0', '-6 days') <> '2024-12-23'
""")
con.commit()
print("sales_gap lines:", query("SELECT COUNT(*) AS n FROM sales_gap")["n"][0], "| sales lines:", len(sales))

### Step 2: Run the query

Two versions of the weekly query on the gap table. The first groups the sales lines alone. The second starts from the calendar, which has all 157 weeks, and attaches the sales to it with a `LEFT JOIN`. A left join keeps every row of the table on its left, whether or not there is a match on the right.

In [ ]:
# Version A: GROUP BY on the sale lines alone
QA = """
SELECT date(SaleDate, 'weekday 0', '-6 days') AS week_start,
       SUM(Quantity)                          AS units
FROM sales_gap
WHERE Category = 'Barbecue and Dining'
GROUP BY week_start
ORDER BY week_start
"""
# Version B: start from the calendar, LEFT JOIN the sales, and turn a missing total into 0
QB = """
SELECT c.WeekStart                  AS week_start,
       COALESCE(SUM(s.Quantity), 0) AS units
FROM calendar AS c
LEFT JOIN sales_gap AS s
       ON date(s.SaleDate, 'weekday 0', '-6 days') = c.WeekStart
      AND s.Category = 'Barbecue and Dining'
GROUP BY c.WeekStart
ORDER BY c.WeekStart
"""
version_a, version_b = query(QA), query(QB)
print("Version A weeks:", len(version_a), "| Version B weeks:", len(version_b))
print(version_b[version_b["week_start"].between("2024-12-16", "2024-12-30")].to_string(index=False))

### Step 3: Read the output in plain English

In [ ]:
missing = sorted(set(version_b["week_start"]) - set(version_a["week_start"]))
print(f"Version A returns {len(version_a)} weeks. Version B returns {len(version_b)}.")
print("The week missing from Version A:", ", ".join(missing))
print("A forecasting model fed Version A would treat the week before and the week after as neighbours. In Version B the gap is a zero, and the model can see it.")

### Step 4: Check against pandas

Check in pandas that the weekly series has no gaps: consecutive week starts should be exactly 7 days apart.

In [ ]:
def gaps_in(series_of_dates):
    days = pd.to_datetime(series_of_dates).diff().dropna().dt.days
    return int((days != 7).sum())

print("Weeks that are not 7 days after the previous one, Version A:", gaps_in(version_a["week_start"]))
print("Weeks that are not 7 days after the previous one, Version B:", gaps_in(version_b["week_start"]))

### Step 5: Say it to a manager

One sentence for the data owner: why does the forecast table start from the calendar and not from the sales?

*Your sentence here*

### Commentary: Part 2

1. What would go wrong in a forecast if the empty week were simply missing?
2. Why `COALESCE(SUM(...), 0)` and not leaving the empty value?

*Your commentary here*

---
## Part 3: Exercise: three questions for Copilot

**The business question:** the buying team has three questions. Ask Copilot to write a query for each, check each one against pandas, and say what you found.

1. Which category sells the most units on a Saturday?
2. In which three weeks of the data did Pots and Planters sell the most units?
3. How many promotion weeks are there in each year, and how many barbecue units were sold in them? (This one needs the calendar.)

### Step 1: Load and look

Know what the answer should look like before you ask. For each question, write down what columns you expect and roughly how many rows.

*Your expectations here*

### Step 2: Run the query

**Copilot:** for each question, give it the table names and column names, then ask for a SQLite query. Paste the query into the cell and run it through `query(...)`. If it uses a function SQLite does not have, tell Copilot "this is SQLite" and try again.

In [ ]:
# Question 1: Copilot's query here
QUESTION_1 = """
"""

In [ ]:
# Question 2: Copilot's query here
QUESTION_2 = """
"""

In [ ]:
# Question 3: Copilot's query here
QUESTION_3 = """
"""

### Step 3: Read the output in plain English

Write one sentence per question: what the answer says, in plain English.

*Your three sentences here*

### Step 4: Check against pandas

Reproduce each answer in pandas. The cell below holds the pandas answers; compare them with your SQL output by eye, then with the comparison cells.

In [ ]:
sales["date"] = pd.to_datetime(sales["SaleDate"])
sales["week_start"] = (sales["date"] - pd.to_timedelta(sales["date"].dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")

pandas_1 = sales[sales["date"].dt.weekday == 5].groupby("Category")["Quantity"].sum().sort_values(ascending=False)
pandas_2 = sales[sales["Category"] == "Pots and Planters"].groupby("week_start")["Quantity"].sum().sort_values(ascending=False).head(3)
promo_weeks = calendar[calendar["PromoActive"] == 1].assign(year=lambda d: d["WeekStart"].str[:4])
bbq_weekly = sales[sales["Category"] == "Barbecue and Dining"].groupby("week_start")["Quantity"].sum()
pandas_3 = promo_weeks.assign(units=promo_weeks["WeekStart"].map(bbq_weekly)).groupby("year").agg(promotion_weeks=("WeekStart", "count"), barbecue_units=("units", "sum"))

print("Question 1 in pandas:\n", pandas_1.to_string(), "\n")
print("Question 2 in pandas:\n", pandas_2.to_string(), "\n")
print("Question 3 in pandas:\n", pandas_3.to_string())

In [ ]:
# Compare your SQL answers with the pandas answers above. Use the names of your own result tables.
# For example: print(list(answer_1["units"]) == list(pandas_1.values))

### Step 5: Say it to a manager

Two sentences for the buying director on whatever surprised you most.

*Your two sentences here*

### Commentary: Part 3

1. Did Copilot get any query wrong first time? What was wrong, and how did the pandas check show it?
2. Question 3 needs the calendar. Why can sales alone not answer it?

*Your commentary here*

**Stretch:** write a query that gives units by category by weekday, as a table with one row per category and one column per weekday. Hint: `SUM(CASE WHEN ... THEN Quantity END)`. Check one cell of it against pandas.

In [ ]:
# Stretch